In [1]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils, boto3, os, utils #paths
import pandas as pd
import sagemaker.core.helper.session_helper as sm_session_helper
import sagemaker.core.image_uris as sm_image_uris
import sagemaker.core.resources as sm_resources
import sagemaker.core.shapes as sm_shapes
from   sagemaker.core.training import configs as sm_train_configs
import sagemaker.core.transformer as sm_transformer
import sagemaker.train.model_trainer as sm_model_trainer
import sagemaker.core.utils.utils as sm_utils

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Existing trusted services: {'states.amazonaws.com', 'sagemaker.amazonaws.com', 'lambda.amazonaws.com', 'events.amazonaws.com'}
Trust policy already up to date for SageMakerExecutionRole-1


[06/26/26 18:09:08] INFO     Found credentials from IAM Role: ec2-1                             ]8;id=14588861;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=14588862;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [39]:
# User vars
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
target_label='rings'
predict_label=target_label+'_prediction'

role = utils.get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sagemaker_session = sm_session_helper.Session(boto_session=boto_session)
sm_client = boto3.client('sagemaker', region_name='us-east-1')
project_dir='s3://omm-test-bucket/abalone-train'
data_dir=f'{project_dir}/data'
model_dir=f'{project_dir}/model'

train_dir=f'{data_dir}/train'
validation_dir=f'{data_dir}/validation'
test_dir=f'{data_dir}/test'
train_file=f'{data_dir}/train/train.csv'
validation_file=f'{data_dir}/validation/validation.csv'
test_file=f'{data_dir}/test/test.csv'

# For development and testing
test_X_file=test_file=f'{data_dir}/test/test_X.csv'
test_y_file=f'{data_dir}/test/test_y.csv'

[06/26/26 17:01:12] INFO     Found credentials from IAM Role: ec2-1                             ]8;id=5365160;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5365161;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
# DATA INGESTION
# sql=utils.Sql('user-1', 'password', 'db_1')
# abalone_df = sql.query('SELECT * FROM abalone;')
# abalone_df=abalone_df.drop(columns=['id','created_at'])

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
cols = ["sex","length","diameter","height","whole_weight",
        "shucked_weight","viscera_weight","shell_weight","rings"]
abalone_df = pd.read_csv(url, header=None, names=cols)

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [ ]:
# FEATURE ENGINEERING
abalone_df = pd.get_dummies(abalone_df, columns=['sex'], drop_first=False)
abalone_df[['sex_I', 'sex_M', 'sex_F']] = abalone_df[['sex_I', 'sex_M', 'sex_F']].astype(int)
abalone_df.head()

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings,sex_F,sex_I,sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,0,0,1
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,0,0,1
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,1,0,0
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,0,0,1
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,0,1,0


In [42]:
# DATA FORMATTING
y = abalone_df['rings']
X = abalone_df.drop(columns=['rings'])

X_train, X_val, X_test, y_train, y_val, y_test = utils.train_val_test_split(X, y, train_size=0.7, val_size=0.15, random_state=42)

pd.concat([y_train, X_train], axis=1).to_csv(train_file, index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(validation_file, index=False, header=False)
pd.concat([y_test, X_test], axis=1).to_csv(test_file, index=False, header=False)
# pd.concat([y_val, X_val], axis=1).to_csv(baseline_file, index=False, header=True) # need header

# For development and testing
X_test.to_csv(test_X_file, index=False, header=False)
y_test.to_csv(test_y_file, index=False, header=False)

input_data_config = [
    sm_train_configs.InputData(channel_name='train', data_source=train_dir, content_type='text/csv'),
    sm_train_configs.InputData(channel_name='validation', data_source=validation_dir, content_type='text/csv')
]

In [11]:
model_trainer = sm_model_trainer.ModelTrainer(
    training_image=sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
    hyperparameters={
        'objective': 'reg:squarederror',
        'num_round': '100',
        'max_depth': '5',
        'eta': '0.2',
        'subsample': '0.8',
        'eval_metric': 'rmse'
    },
    role=role,
    sagemaker_session=sagemaker_session,
    base_job_name='abalone-train-job',
    output_data_config=sm_shapes.OutputDataConfig(s3_output_path=model_dir),
    networking=sm_train_configs.Networking(
        subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
        security_group_ids=['sg-00f14515abe1e47e8']
    )
)

[06/25/26 21:35:49] INFO     Ignoring unnecessary instance type: None.                            ]8;id=2330266;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=2330267;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     Compute not provided. Using default:                                   ]8;id=2330274;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2330275;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py#102\102]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=2330281;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2330282;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=2330289;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=2330290;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.                     
                             5-1                                                                                   

In [12]:
training_job = model_trainer.train(
    input_data_config=input_data_config,
    wait=True,
    logs=True
)

[06/25/26 21:35:53] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2330297;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2330298;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=2330305;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2330306;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31116\31116]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=2330313;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=2330314;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=2330320;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=2330321;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/utils/utils.py#375\375]8;;\

...

                    INFO     abalone-train-job-20260625213553/algo-1-1782423394:                 ]8;id=2331035;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2331036;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [96]#011train-rmse:1.25616#011validation-rmse:2.02881                                 

                    INFO     abalone-train-job-20260625213553/algo-1-1782423394:                 ]8;id=2331041;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2331042;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [97]#011train-rmse:1.25277#011validation-rmse:2.02753                                 

                    INFO     abalone-train-job-20260625213553/algo-1-1782423394:                 ]8;id=2331047;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2331048;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [98]#011train-rmse:1.24839#011validation-rmse:2.02505                                 

                    INFO     abalone-train-job-20260625213553/algo-1-1782423394:                 ]8;id=2331053;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2331054;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31439\31439]8;;\
                             [99]#011train-rmse:1.23891#011validation-rmse:2.02563                                 

[06/25/26 21:38:16] INFO     Final Resource Status: Completed                                    ]8;id=2331060;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2331061;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/resources.py#31442\31442]8;;\

## Resister for approval

In [ ]:
def register_new_model_version(model_package_group_name, model_data_url, image, inference_types=['ml.m5.large', 'ml.m5.xlarge'], tranform_types=['ml.m5.large', 'ml.m5.xlarge'], context_types=['text/csv'], response_mime_types=['text/csv']):
    sm_client = boto3.client('sagemaker')
    model_package_exists = False
    groups = sm_client.list_model_package_groups()
    for item in groups['ModelPackageGroupSummaryList']:
        if item['ModelPackageGroupName'] == model_package_group_name:
            print('Using existing ModelPackageGroupName')
            model_package_exists = True
            break
    if not model_package_exists:
        print('Making new ModelPackageGroupName')
        sm_client.create_model_package_group(ModelPackageGroupName=model_package_group_name)

    response = sm_client.create_model_package(
        ModelPackageGroupName='abalone',  # use group name, not ModelPackageName
        InferenceSpecification={
            'Containers': [
                {
                    'Image': image,
                    'ModelDataUrl': model_data_url
                }
            ],
            'SupportedContentTypes': context_types,
            'SupportedResponseMIMETypes': response_mime_types,
            'SupportedRealtimeInferenceInstanceTypes': inference_types,
            'SupportedTransformInstanceTypes': tranform_types
        },
        ModelApprovalStatus='PendingManualApproval'
    )

    return response['ModelPackageArn']

abalone


/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
register_new_model_version(
    'abalone', 
    's3://omm-test-bucket/abalone-train/model/abalone-train-job-20260625213553/output/model.tar.gz', 
    sm_image_uris.retrieve('xgboost', region, version='1.5-1')
)

In [ ]:
utils.view_model_versions(boto_session)

In [68]:
import pandas as pd
import json, datetime
sm_runtime = boto3.client('sagemaker-runtime')
s3_client=boto3.client('s3')

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [ ]:
row_nums=list(range(0,5)) # inclusive , exclusive

test_X=pd.read_csv('s3://omm-test-bucket/abalone-train/data/test/test_X.csv', header=None)

for row_num in row_nums:
    assn_id=row_num
    row=test_X.iloc[row_num]
    input = str(list(row)).replace(' ','').replace(']','').replace('[','')
    print(input)
    response = sm_runtime.invoke_endpoint(EndpointName="abalone-1-endpoint", ContentType="text/csv", Body=input, InferenceId=str(row_num))
    print(response["Body"].read().decode("utf-8"))


0.255,0.19,0.07,0.0815,0.028,0.016,0.031,0.0,1.0,0.0
{'ResponseMetadata': {'RequestId': '3dab1e19-317b-45ce-870f-7c4039389bf0', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '3dab1e19-317b-45ce-870f-7c4039389bf0', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Fri, 26 Jun 2026 17:28:26 GMT', 'content-type': 'text/csv; charset=utf-8', 'content-length': '18', 'connection': 'keep-alive'}, 'RetryAttempts': 0}, 'ContentType': 'text/csv; charset=utf-8', 'InvokedProductionVariant': 'AllTraffic', 'Body': <botocore.response.StreamingBody object at 0x7f75b7935790>}
0.63,0.48,0.145,1.0115,0.4235,0.237,0.305,0.0,0.0,1.0
{'ResponseMetadata': {'RequestId': 'd81c4ad1-ebba-4e10-ae7d-4957b784b050', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'd81c4ad1-ebba-4e10-ae7d-4957b784b050', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Fri, 26 Jun 2026 17:28:26 GMT', 'content-type': 'text/csv; charset=utf-8', 'content-length': '19', 'connection': 'keep-alive

In [72]:
row_nums=list(range(0,5)) # inclusive , exclusive

test_y=pd.read_csv('s3://omm-test-bucket/abalone-train/data/test/test_y.csv', header=None)
gt_bucket='omm-test-bucket'
gt_dir_key='abalone-ground-truth'

gt_data=[]
for i in range(0, len(row_nums)):
    row_num=row_nums[i]
    assn_id=str(row_num)
    # gt_val=gt_vals[i]
    gt_val=str(list(test_y.iloc[row_num])[0])
    gt_row={"groundTruthData": {"data": gt_val, "encoding": "CSV"}, "eventMetadata": {"eventId": assn_id}, "eventVersion": "0"}
    #print(gt_row)
    gt_data.append(json.dumps(gt_row).replace("'", '"'))
payload = "\n".join(gt_data)
print(payload)
# S3 key must match the data capture path structure
now = datetime.datetime.utcnow()
s3_client.put_object(
    Bucket=gt_bucket,
    Key=f"{gt_dir_key}/{now.year}/{now.month:02d}/{now.day:02d}/{now.hour:02d}{now.minute:02d}.jsonl",
    Body=payload.encode("utf-8")
)

# {"groundTruthData": {"data": "1", "encoding": "CSV"}, "eventMetadata": {"eventId": "my-row-id-101"}, "eventVersion": "0"}
# {"groundTruthData": {"data": "0", "encoding": "CSV"}, "eventMetadata": {"eventId": "my-row-id-202"}, "eventVersion": "0"}

{"groundTruthData": {"data": "5", "encoding": "CSV"}, "eventMetadata": {"eventId": "0"}, "eventVersion": "0"}
{"groundTruthData": {"data": "12", "encoding": "CSV"}, "eventMetadata": {"eventId": "1"}, "eventVersion": "0"}
{"groundTruthData": {"data": "14", "encoding": "CSV"}, "eventMetadata": {"eventId": "2"}, "eventVersion": "0"}
{"groundTruthData": {"data": "8", "encoding": "CSV"}, "eventMetadata": {"eventId": "3"}, "eventVersion": "0"}
{"groundTruthData": {"data": "11", "encoding": "CSV"}, "eventMetadata": {"eventId": "4"}, "eventVersion": "0"}


{'ResponseMetadata': {'RequestId': 'SFSC96EW33GKNZ4Z',
  'HostId': '9IznEWuToXtZNCUl/3QBWwPzfBsWtonPF8ZBxpywywZDqOQH32kuLAzDtNzVBsMUzKT17nfuuPM=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '9IznEWuToXtZNCUl/3QBWwPzfBsWtonPF8ZBxpywywZDqOQH32kuLAzDtNzVBsMUzKT17nfuuPM=',
   'x-amz-request-id': 'SFSC96EW33GKNZ4Z',
   'date': 'Fri, 26 Jun 2026 17:47:07 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"dbe6daaf88143dc085264482859b7a7c"',
   'x-amz-checksum-crc32': 'hsKyHw==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"dbe6daaf88143dc085264482859b7a7c"',
 'ChecksumCRC32': 'hsKyHw==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [74]:
sm = boto3.client("sagemaker")

# list all schedules
schedules = sm.list_monitoring_schedules()
for s in schedules["MonitoringScheduleSummaries"]:
    print(s["MonitoringScheduleName"], s["MonitoringScheduleStatus"])

# get details on a specific schedule
response = sm.describe_monitoring_schedule(
    MonitoringScheduleName="abalone-1-endpoint-mb-mon"
)
print(response["MonitoringScheduleStatus"])   # Scheduled, Stopped, Pending
print(response["MonitoringScheduleConfig"])   # cadence, output path, etc.

abalone-1-endpoint-mb-mon Scheduled
abalone-1-endpoint-dq-mon Scheduled
abalone-1-endpoint-mq-mon Scheduled
abalone-1-endpoint-me-mon Scheduled
Scheduled
{'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)', 'DataAnalysisStartTime': '-PT2H', 'DataAnalysisEndTime': '-PT1H'}, 'MonitoringJobDefinitionName': 'abalone-1-endpoint-mb-mon-job', 'MonitoringType': 'ModelBias'}


In [75]:
executions = sm.list_monitoring_executions(
    MonitoringScheduleName="abalone-1-endpoint-mb-mon",
    SortBy="ScheduledTime",
    SortOrder="Descending",
    MaxResults=10
)

for e in executions["MonitoringExecutionSummaries"]:
    print(e["ScheduledTime"], e["MonitoringExecutionStatus"])

2026-06-26 17:00:00+00:00 Failed
2026-06-26 16:00:00+00:00 Failed
